### Setup

Loading packages and texts, and preparing Pydantic classes

In [27]:
import pandas as pd
import numpy as np
import ollama
import re
import json

from src.io import read_tabular
from pathlib import Path

from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Literal

## Loading data ##
# current dataset: UK newspaper data
fp = Path("../../data")
df_uk = read_tabular(fp / "UK_texts.csv")
input = df_uk['contexted'][0:3]

## Pydantic models

## Model for entity extraction
class Entity(BaseModel):
    verbatim: Optional[str] = Field(None, description = "The entity as it is referenced in the text")
    organisation: Optional[str] = Field(None, description = "The organisation the subject belongs to")
    individual: Optional[str] = Field(None, description = "The individual mentioned in the text")

entity_json_schema = Entity.model_json_schema()

## Model for relations
class Claim(BaseModel):
    actor_verbatim: Optional[str] = Field(None, description = "The actor as referenced in the text")
    actor_organisation: Optional[str] = Field(None, description = "The organisation the actor belongs to")
    actor_individual: Optional[str] = Field(None, description = "The individual mentioned in the text")
    target: Optional[str] = Field(None, description = "The actor or issue the claim is about")

claim_schema = Claim.model_json_schema()

## Model for issue categories
class Issue(BaseModel):
    issue: str = Field(..., description="The issue as it is referenced in the text")
    issue_cat: str = Field(..., description="The issue category according to the codebook")

## Final core sentence  model
class CoreSent(BaseModel):
    type: Literal['actor-actor', 'actor-issue', 'NA'] = Field(..., description = "The category of core sentence detected")
    subject: str = Field(..., description="The subject as it appears in the core sentence")
    subject_organisation: str = Field(..., description="The subject_organisation of the core sentence")
    direction: Literal["support", "opposition", "ambivalent", 'NA'] = Field(..., description = "The stance taken by the actor towards the subject")
    object: Optional[str] = Field(None, description = "The object as it appears in the core sentence")
    object_organisation: Optional[str] = Field(None, description = "The object_organisation of the core sentence")
    issue: Optional[str] = Field(None, description = "An issue being referenced in the core sentence")
    issue_cat: Optional[str] = Field(None, description = "The issue category according to the codebook")

class CSResponse(BaseModel):
    sentence: str = Field(..., description="The grammatical sentence you coded")
    core_sents: Optional[List[CoreSent]] = Field(
        None,
        description="List of core sentences extracted from the sentence. Leave empty if none are detected."
    )

### Prompts

Needs:
- NER prompt
- Determine political statement
- Identify target(s) + type
    - Might do as two steps:
        - Does the actor target another actor?
        - Does the actor speak about an issue?
- Classify issues
- Identify direction
- Identify issue reference if required
- Reviewer?

In [ ]:
ner_prompt = f'''
You are an expert annotator for political texts. Your task is to identify political actors and their organisations from articles published in British newspapers between November 2025 and February 2026. Use your background knowledge of British politics in this time period to identify actors and the organisations they are affiliated to.

## Instructions

You will receive a sequence of five sentences from a British newspaper article. One of those sentences is marked with > and <. Focus on the marked sentence, and use the rest of the text as context to help you identify actors that are referred to indirectly. For example, if the marked sentence contains a pronoun, you should use the context to identify the actor that the pronoun refers to. Extract all actors mentioned in the marked sentence that are affiliated with a political party, civil society movement, organised business interest, experts, or other political actors. You should identify and extract any mention of an individual politician, public figure, expert, etc., as well as any mention of organised groups.

## Variables

For each actor detected in the sentence, provide the following information:
- "verbatim": The actor as it is referenced in the sentence. This can be a pronoun, a name, or a description. If an actor is not referenced explicitly, but is making a statement (e.g. as in "The Prime Minister said: > [statement] <"), write "Implicit". 
- "organisation": The organisation the actor is affiliated with. If there are multiple possible affiliations, choose the one that is most relevant to the context of the sentence. If the actor is affiliated with the government, choose the party or parties in government. If the actor is not affiliated with an organisation, return "Independent [type of actor], e.g. "Independent expert" for a scientist.
- "individual": If the actor is an individual, provide their name in the format "[Last Name], [First Name]". If the actor is not an individual, leave this field empty.

## Output Format

Return a JSON list with an entry for each detected actor following this scheme: {entity_json_schema}. 

Return nothing else. Do not include any additional text or explanations. Do not wrap in a code block.

## Example

Input: "Yesterday, the Chancellor of the Exchequer, Rachel Reeves, presented the new budget in Parliament. Chancellor Reeves highlighted the government's commitment to climate action. It has set aside £20 billion for green energy projects. > Reeves faced criticism from the opposition for ignoring the challenges of energy prices for households, something that experts have warned could lead to increased energy poverty. < Reform went further, calling for a complete end to the government's Net Zero policy."

Sentence to analyse: "Reeves faced criticism from the opposition for ignoring the challenges of energy prices for households, something that experts have warned could lead to increased energy poverty."

Output: [{{"verbatim": "Reeves", "organisation": "Labour Party", "individual": "Reeves, Rachel"}}, {{"verbatim": "the opposition", "organisation": "Conservative Party", "individual": ""}}, {{"verbatim": "experts", "organisation": "Independent experts", "individual": ""}}]
'''

In [25]:
statement_prompt = f'''You are an expert annotator for political texts. Your task is to identify political claims made by actors in articles published in British newspapers between November 2025 and February 2026. Rely only on the text provided to you, and do not use external information such as the positions of actors you know. Use the following instructions to identify political claims:

## Input

You will be given a sequence of five sentences from a newspaper article. One of those sentences is marked with > and <. Focus on the marked sentence, and use the rest of the text as context to help you identify political claims made by actors in the marked sentence. You will also be given an actor that has been identified in the sentence. Your task is to only focus on the provided actor, and identify any claims made by that specific actor. Ignore any statements potentially made by other actors in the sentence.

## Instructions

Political claims are statements made by actors that express a position on a political actor or issue. Whenever a subject takes a stance towards a target, or takes an action against a target, there is a relation between them. The subject is always the actor that is taking the action or stance, even if they are not the grammatical subject of the sentence. Note that each actor may make multiple claims in a single sentence.Follow these steps to identify whether there is an actor:
1) Find the provided actor in the sentence. It should always be referred to in the sentence with the "verbatim" value provided. If the "verbatim" value is "Implicit", then the actor is not explicitly mentioned in the sentence, but is making a statement. In this case, treat the speaker as the actor.
2) Identify the verbs in the sentence and determine whether the actor acts as a semantic subject for any of the verbs in the sentence. If the actor is not a semantic subject for any of the verbs, then there is no claim made by the actor in this sentence.
3) If the actor is a semantic subject for one or more verbs, then identify the object of the verb. The object can be a noun phrase, a pronoun, or an entire clause. If there is no object, then there is no claim made by the actor in this sentence.
4) Determine whether an opinion or stance of the actor is being described or stated, or if the actor is just being mentioned by the writer. If the actor is taking a stance, then the actor is making a claim. If the actor is just being described, then there is no claim made by the actor in this sentence. Consequences of prior actions are not claims, unless the actor is taking a stance on the action.

## Output

Return a JSON with a new entry for each claim you found, using the following variables:
- "verbatim": The verbatim actor as it was provided to you in the input.
- "organisation": The organisation the actor is affiliated with, as it was provided to you in the input.
- "individual": The individual name of the actor, as it was provided to you in the input.
- "target": The target of the claim, which can be a political actor or an issue. Quote the target issue verbatim, as it was mentioned in the marked sentence. Do not reference targets that are simply present in the context sentences, unless they are mentioned in the marked sentence as well.

If you did not find a claim for a given actor, return an empty list. Return nothing else. Do not include any additional text or explanations. Do not wrap in a code block.
'''

## Define Ollama inference function

Repeatable function to call different prompts.

In [32]:
def inference(sysprompt, userprompt, model, options, format, client):

    message = [
        {"role": "system", "content": sysprompt},
        {"role": "user", "content": userprompt}
    ]

    response = client.chat(model = model, 
                           messages = message, 
                           options = options,
                           format = format)

    out = response.get("message", {}).get("content", "{}")

    return out

def verification(raw, pyd_scheme):
    val_out = []
    parsed = json.loads(raw)

    for item in parsed:
        try: 
            validated = pyd_scheme(**item)
            val_out.append(validated.model_dump())
        except (ValidationError) as e:
            print(f"Invalid response: {raw}")
        except (json.JSONDecodeError) as e:
            print(f"Not a valid JSON: {raw}")

    return val_out

## Inference setup

Model name, options, pulling, client setup

In [21]:
GEMMA_CLOUD = "gemma4:31b-cloud"

modelname = GEMMA_CLOUD

opts = {
    "seed": 42,
    "temperature": 0.0
}

ollama.pull(modelname)
client = ollama.Client()

## Inference Pipeline

In [ ]:
for text in input:
    entity_raw = inference(ner_prompt, text, modelname, opts, "json", client)
    entities = json.loads(entity_raw)

    print(f"Text: {text}")
    print(f"Entities: {entities}")

    claims = []

    for entity in entities:
        input_prompt = f'''
        {text}

        Identify all political claims according to the provided instructions. Focus only on the provided actor: {entity}.
        '''

        claims_raw = inference(statement_prompt, input_prompt, modelname, opts, claim_schema, client)

        # Verification
        val_claims = verification(claims_raw, Claim)
        claims.extend(val_claims)

    

Text: Among those measures, according to sources briefed on the budget preparations, is a plan to take energy efficiency levies off bills and fund them through the government’s existing warm homes plan. The move will mean restricting heat-pump subsidies so that only those receiving certain benefits will be allowed to claim them, sharply bringing down costs to the government. Supporters of the change say that the subsidies, which can be as high as £7,500, were largely going to middle-class households that could have afforded them anyway > Energy industry experts, however, warn that by taking the support away ministers will slow the transition from gas boilers to more expensive but cleaner heat pumps. < Sam Alvis, the head of energy and environment at the Institute of Public Policy Research thinktank, said: “The urge to get bills down is the right one, everything should be on the table.”
Entities: [{'verbatim': 'Energy industry experts', 'organisation': 'Independent experts', 'individual